# 🏦 Pipeline Preditivo de Risco de Crédito

**Curso:** Machine Learning e Visão Computacional [T5] — SENAI/SC
**Autor:** Waldinei Lameira Rosa
**Base escolhida:** Opção A — Risco de Crédito (Setor Financeiro)

---

## Objetivo de negócio

Prever a probabilidade de um cliente se tornar **inadimplente** (`loan_status = 1`) em comparação aos bons pagadores (`loan_status = 0`), comparando dois algoritmos clássicos — **KNN** e **Árvore de Decisão** — e recomendando o mais adequado para produção com base no **impacto financeiro** dos erros:

- **Falso Positivo (FP):** bom pagador rotulado como "risco de calote" → o banco nega crédito e perde a receita de juros.
- **Falso Negativo (FN):** mau pagador rotulado como "seguro" → o banco concede crédito e pode perder o capital emprestado.

## Roteiro do notebook

| Fase | Conteúdo |
|---|---|
| 0 | Configuração do ambiente e carregamento da base |
| 1 | Análise Exploratória de Dados (EDA) |
| 2 | Tratamento e limpeza (duplicados, nulos, outliers) |
| 3 | Feature Engineering — `comprometimento_renda` |
| 4 | Encoding, split estratificado, balanceamento e escalonamento seguro |
| 5 | Modelagem e diagnóstico de overfitting (KNN × Árvore) |
| 6 | Avaliação e veredito de negócio |

---
# Fase 0 — Configuração do ambiente

**Decisões técnicas desta fase:**

1. **Importações centralizadas no topo:** todas as bibliotecas usadas no pipeline são importadas em uma única célula. Se alguma estiver faltando no `.venv`, o erro aparece logo no início, e não no meio da modelagem.
2. **Semente aleatória fixa (`SEED = 42`):** o split treino/teste, o SMOTE e a Árvore de Decisão envolvem aleatoriedade. Fixar a semente garante **reprodutibilidade** — qualquer pessoa que executar o notebook obterá exatamente os mesmos números apresentados no README e no vídeo.
3. **Caminhos relativos com `pathlib`:** o notebook funciona em qualquer máquina (Windows, Linux ou macOS) sem precisar editar caminhos absolutos.

In [1]:
# ============================================================
# 0.1 Importação das bibliotecas
# ============================================================

# --- Sistema e utilidades ---
from pathlib import Path              # manipulação de caminhos independente do sistema operacional
import warnings                       # controle de mensagens de aviso

# --- Manipulação de dados ---
import numpy as np                    # operações numéricas e vetoriais
import pandas as pd                   # estrutura tabular (DataFrame)

# --- Visualização ---
import matplotlib.pyplot as plt       # base gráfica
import seaborn as sns                 # gráficos estatísticos de alto nível

# --- Pré-processamento (scikit-learn) ---
from sklearn.model_selection import train_test_split   # divisão treino/teste estratificada
from sklearn.preprocessing import StandardScaler       # padronização (z-score) — usada só no KNN

# --- Balanceamento de classes (imbalanced-learn) ---
from imblearn.over_sampling import SMOTE               # sobreamostragem sintética da classe minoritária

# --- Modelos ---
from sklearn.neighbors import KNeighborsClassifier     # KNN: baseado em distância euclidiana
from sklearn.tree import DecisionTreeClassifier        # Árvore: baseada em cortes por limiar

# --- Métricas de avaliação ---
from sklearn.metrics import (
    accuracy_score,            # proporção de acertos totais
    precision_score,           # dos previstos como inadimplentes, quantos realmente são
    recall_score,              # dos inadimplentes reais, quantos o modelo capturou
    f1_score,                  # média harmônica entre precisão e recall
    classification_report,     # relatório completo por classe
    confusion_matrix,          # matriz VN / FP / FN / VP
    ConfusionMatrixDisplay,    # plot da matriz de confusão
)

print("✅ Bibliotecas importadas com sucesso.")

✅ Bibliotecas importadas com sucesso.


In [2]:
# ============================================================
# 0.2 Configurações globais do projeto
# ============================================================

# Semente aleatória única, reutilizada em TODAS as etapas com aleatoriedade
# (split, SMOTE e Árvore de Decisão) para garantir reprodutibilidade.
SEED = 42

# Caminhos relativos: o notebook fica em notebooks/, então subimos um nível (..)
# para chegar à raiz do projeto.
RAIZ_PROJETO = Path("..").resolve()
CAMINHO_DADOS = RAIZ_PROJETO / "data" / "raw" / "credit_risk_dataset.csv"
PASTA_FIGURAS = RAIZ_PROJETO / "reports" / "figures"
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)  # cria a pasta se ainda não existir

# Estilo visual padronizado para todos os gráficos do relatório
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)   # tamanho padrão das figuras
plt.rcParams["figure.dpi"] = 100           # resolução adequada para tela e README

# Exibição do pandas: mostrar todas as colunas e 2 casas decimais
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Oculta avisos de depreciação que não afetam o resultado (mantém o notebook limpo)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Raiz do projeto : {RAIZ_PROJETO}")
print(f"Arquivo de dados: {CAMINHO_DADOS}")

Raiz do projeto : C:\Users\waldinei.rosa\Projetos\risco-credito-ml
Arquivo de dados: C:\Users\waldinei.rosa\Projetos\risco-credito-ml\data\raw\credit_risk_dataset.csv


In [3]:
# ============================================================
# 0.3 Carregamento da base de dados
# ============================================================

# Verificação defensiva: se o CSV não estiver no lugar esperado,
# interrompemos com uma mensagem clara em vez de um erro genérico.
if not CAMINHO_DADOS.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado em {CAMINHO_DADOS}.\n"
        "Baixe a base do link do enunciado e salve em data/raw/credit_risk_dataset.csv"
    )

# Leitura do CSV original. Guardamos em df_original (intocado) e trabalhamos
# em uma cópia (df), para sempre podermos comparar o antes e o depois da limpeza.
df_original = pd.read_csv(CAMINHO_DADOS)
df = df_original.copy()

print(f"✅ Base carregada: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

✅ Base carregada: 32,581 linhas × 12 colunas


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.00,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.00,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.00,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.00,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.00,MEDICAL,C,35000,14.27,1,0.55,Y,4
